In [23]:
import yaml 
from pathlib import Path

import mlflow

from pytorch_pipeline import val
from pytorch_pipeline.train import build_pipeline_dataloaders, build_datasets, get_device
from pytorch_pipeline.utils import resolve_uri, Config, resolve_hardware_profile, get_current_git_branch, CLASS_ORDER
from pytorch_pipeline.utils.params import DatasetParams, DataLoadersParams, PathsParams
from pytorch_pipeline.review import review_label_issues


In [24]:
#Args
test = False
test_fraction = 0.05
seed = 42
model_name = 'cv_pheno_inat'
model_version = 1
config_path = Path("/home/etienne/projects/inat-phenology-cv/configs/local.yaml")

In [25]:
# Set up environment specific configs
with open(config_path, "r") as file:
    env_configs = yaml.safe_load(file)
paths_params = PathsParams(**env_configs["paths"])
dataloader_params = DataLoadersParams(**env_configs["dataloader_params"])
hardware_profile = resolve_hardware_profile()
configs = Config(
    config_path,
    paths_params=paths_params,
    dataloaders_params=dataloader_params,
    hardware_profile=hardware_profile,
    git_branch=get_current_git_branch(),
    )

In [26]:
# Load the model
mlflow.set_tracking_uri(resolve_uri())
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.pytorch.load_model(model_uri)

In [27]:
#Load model, dataset & dataloaders 

configs.test = test
device = get_device()
dataset_params = DatasetParams(testing_frac=test_fraction)
configs.dataset_params = dataset_params
datasets = build_datasets(configs, model, seed= seed)
_, val_loader, _ = build_pipeline_dataloaders(datasets, configs.dataloaders_params, seed=seed)

Running on cuda


In [28]:
#Run inference
obs_ids, raw_labels, raw_preds = val.execute(model=model, dataloader=val_loader, device=device, as_numpy=True)

In [29]:
from cleanlab.filter import find_label_issues
import numpy as np

means = []
issues = []

for i in range(3):
    # Format predicted probs for cleanlab
    labels = raw_labels[:,i].astype(int)
    pred_probs_pos = raw_preds[:,i]
    pred_probs_neg = 1 - pred_probs_pos
    pred_probs = np.column_stack((pred_probs_neg, pred_probs_pos))
    issue_mask = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    )
    means.append(issue_mask.mean())

    ordered_issue_indices = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
    )
    
    issues.append([obs_ids[i] for i in ordered_issue_indices])

In [30]:
for i, _ in enumerate(CLASS_ORDER):
    print(f"{CLASS_ORDER[i]} {means[i]}")
    print(f" {len(issues[i])} issues")

Flowering 0.11548556430446194
 176 issues
Fruiting 0.06955380577427822
 106 issues
Flower_Budding 0.1620734908136483
 247 issues


In [ ]:
class_idx = 2

review_label_issues(obs_ids= issues[class_idx],
 label_name= CLASS_ORDER[class_idx],
 db_path= "/home/etienne/projects/inat-phenology-cv/data/cv_raw.duckdb",
 image_dir= "/home/etienne/projects/inat-phenology-cv/data/images",
 all_obs_ids=obs_ids,
 raw_labels=raw_labels,
 raw_preds=raw_preds,
 class_idx=class_idx
 )

[{'obs_id': 228564603,
  'weights': [],
  'prob': 0.0145263671875,
  'target': 1,
  'paths': ['/home/etienne/projects/inat-phenology-cv/data/images/405609407.jpg']},
 {'obs_id': 100434912,
  'weights': [],
  'prob': 0.027587890625,
  'target': 1,
  'paths': ['/home/etienne/projects/inat-phenology-cv/data/images/167626501.jpg',
   '/home/etienne/projects/inat-phenology-cv/data/images/167626558.jpg']},
 {'obs_id': 259676182,
  'weights': [],
  'prob': 0.037841796875,
  'target': 1,
  'paths': ['/home/etienne/projects/inat-phenology-cv/data/images/466195074.jpg',
   '/home/etienne/projects/inat-phenology-cv/data/images/466195095.jpg',
   '/home/etienne/projects/inat-phenology-cv/data/images/466195134.jpg',
   '/home/etienne/projects/inat-phenology-cv/data/images/466195151.jpg',
   '/home/etienne/projects/inat-phenology-cv/data/images/466195182.jpg']},
 {'obs_id': 106683962,
  'weights': [],
  'prob': 0.0419921875,
  'target': 1,
  'paths': ['/home/etienne/projects/inat-phenology-cv/data/i